# Crop Recommendation System — Data Analysis & Model Training

This notebook covers:
1. **Dataset sources**: Zenodo (Nigeria field data 5k), Kaggle (India 2.2k), SF24 (California 2.2k)
2. **Merging & cleaning**: Normalize schemas, filter low-sample crops, build region maps
3. **EDA**: Feature distributions, correlations, crop-wise statistics
4. **Model training**: XGBoost with cross-validation
5. **Results**: Accuracy, feature importance, per-class metrics

In [ ]:
#load data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

BASE = Path.cwd()

## 1. Dataset Sources

### Zenodo (Primary — Real Field Data)
- **Source**: Kwaghtyo et al. 2026, DOI: 10.5281/zenodo.19709807
- **Location**: Yandev District, Gboko LGA, Benue State, Nigeria
- **Collection**: Field surveys across two farming seasons (2024-2025)
- **Size**: 5,000 samples, 10 crops
- **Real soil lab analysis**: Samples collected at 0-30cm depth, analyzed in lab
- **Crops**: maize, rice, pepper, soybean, beans, orange, guinea corn, cassava, tomatoes, yam

### Kaggle (Secondary — Indian Agricultural Data)
- **Source**: Atharva Ingle, Indian Chamber of Food and Agriculture
- **Size**: 2,200 samples, 22 crops × 100 each
- **Note**: Used only to boost overlapping crops (maize, rice, orange)

### SF24 (California)
- **Source**: Kaggle Smart Farming Data 2024
- **Size**: 2,200 rows, 22 crops — same core data as Kaggle with extra features
- **Used for**: Extra feature exploration (soil_type, soil_moisture, etc.)

**Merge strategy**: Zenodo + Kaggle → filtered to crops with ≥200 samples → 5,300 rows × 10 crops

In [ ]:
# Load the merged dataset
df = pd.read_csv(BASE / 'dataset' / 'Crop_recommendation_merged.csv')
df['label'] = df['label'].str.strip().str.lower()
print(f"Shape: {df.shape}")
print(f"Crops: {df['label'].nunique()}")
print(f"Features: {list(df.columns)}")

In [ ]:
print(df.head())
df.info()
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print(f"Duplicates: {df.duplicated().sum()}")

In [ ]:
# Crop distribution
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True))

## 2. Exploratory Data Analysis

In [ ]:
# Feature distributions by crop
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
for idx, feat in enumerate(features):
    ax = axes[idx // 2, idx % 2]
    for crop in sorted(df['label'].unique()):
        subset = df[df['label'] == crop][feat]
        ax.hist(subset, bins=15, alpha=0.5, label=crop)
    ax.set_title(f'{feat} Distribution by Crop')
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7, ncol=2)
axes[3, 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df[features].corr()
sns.heatmap(corr, annot=True, cmap='RdYlBu', vmin=-1, vmax=1, square=True)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Mean feature values per crop
grouped = df.groupby('label')[features].mean().round(2)
print(grouped)

In [ ]:
# Box plots for each feature
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for idx, feat in enumerate(features):
    ax = axes[idx // 3, idx % 3]
    sns.boxplot(x='label', y=feat, data=df, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{feat}')
axes[2, 2].axis('off')
plt.tight_layout()
plt.show()

## 3. Region Mapping

Crops are mapped to regions based on dataset origin. This mapping is used at inference time to filter predictions.

In [ ]:
with open(BASE / 'crop_region_map.json') as f:
    region_map = json.load(f)
for crop, regions in sorted(region_map.items()):
    count = len(df[df['label'] == crop])
    print(f"{crop:15s} ({count:4d} samples) → {regions}")

## 4. Soil Type Profiles

Computed from Zenodo data's `soil` column. These profiles auto-fill NPK/pH values when a user selects their soil type.

In [ ]:
with open(BASE / 'soil_profiles.json') as f:
    profiles = json.load(f)
for st, p in sorted(profiles.items()):
    print(f"{st:12s}: N={p['N_median']:>5.1f}  P={p['P_median']:>5.1f}  K={p['K_median']:>5.1f}  pH={p['ph_median']:.2f}  ({p['samples']} samples)")

## 5. Model Training

Training XGBoost on the merged dataset with 7 features (no region — region is used post-prediction as a filter).

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[features].values
le = LabelEncoder()
y = le.fit_transform(df['label'])

print("Label mapping:")
for i, label in enumerate(le.classes_):
    count = len(df[df['label'] == label])
    print(f"  {i}: {label} ({count} samples)")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

### 5.1 Multi-Model Comparison (same as original notebook)

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                             gamma=0.2, reg_lambda=3.0, reg_alpha=1.0,
                             random_state=42, eval_metric='mlogloss', verbosity=0),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    results.append({'Model': name, 'Train Acc': f'{train_acc:.4f}', 'Val Acc': f'{val_acc:.4f}'})
    print(f"{name:20s}  Train: {train_acc:.4f}  Val: {val_acc:.4f}")

pd.DataFrame(results)

In [ ]:
# Cross-validation on XGBoost
cv_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                         subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                         gamma=0.2, reg_lambda=3.0, reg_alpha=1.0,
                         random_state=42, eval_metric='mlogloss', verbosity=0)
cv_scores = cross_val_score(cv_model, X, y, cv=5, scoring='accuracy')
print(f"CV scores: {cv_scores}")
print(f"CV mean:   {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

In [ ]:
# Final XGBoost model
final_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                            subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                            gamma=0.2, reg_lambda=3.0, reg_alpha=1.0,
                            random_state=42, eval_metric='mlogloss', verbosity=0)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_val)
print(classification_report(y_val, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix (XGBoost)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Feature importance
importance = final_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importance})
feat_imp = feat_imp.sort_values('Importance', ascending=False)
print(feat_imp)

plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Save the model
joblib.dump(final_model, 'xgboost_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Model saved: xgboost_model.pkl")
print("Encoder saved: label_encoder.pkl")

In [ ]:
# Quick prediction test
test_input = np.array([[60, 30, 40, 25.0, 75.0, 6.5, 120.0]])
pred = final_model.predict(test_input)
probs = final_model.predict_proba(test_input)[0]
crop = le.inverse_transform(pred)[0]
conf = probs[pred[0]] * 100
print(f"Test input: N=60, P=30, K=40, Temp=25°C, Humidity=75%, pH=6.5, Rainfall=120mm")
print(f"Predicted: {crop} (confidence: {conf:.1f}%)")
print("\nTop 3 predictions:")
for i in np.argsort(probs)[-3:][::-1]:
    print(f"  {le.classes_[i]}: {probs[i]*100:.1f}%")

In [ ]:
# Save crop range summary for LLM context
crop_ranges = {}
for crop in sorted(df['label'].unique()):
    sub = df[df['label'] == crop]
    crop_ranges[crop] = {
        'N': [int(sub['N'].min()), int(sub['N'].max())],
        'P': [int(sub['P'].min()), int(sub['P'].max())],
        'K': [int(sub['K'].min()), int(sub['K'].max())],
        'temperature': [round(float(sub['temperature'].min()), 1), round(float(sub['temperature'].max()), 1)],
        'humidity': [round(float(sub['humidity'].min()), 1), round(float(sub['humidity'].max()), 1)],
        'ph': [round(float(sub['ph'].min()), 2), round(float(sub['ph'].max()), 2)],
        'rainfall': [round(float(sub['rainfall'].min()), 1), round(float(sub['rainfall'].max()), 1)],
    }

with open('crop_ranges.json', 'w') as f:
    json.dump(crop_ranges, f, indent=2)
print(f"crop_ranges.json saved with {len(crop_ranges)} crops")

## Summary

- **Dataset**: 5,300 real field samples (Zenodo Nigeria + Kaggle India)
- **Crops**: 10 well-sampled crops (≥200 samples each)
- **Features**: N, P, K, temperature, humidity, pH, rainfall
- **Model**: XGBoost
- **Accuracy**: 100% on validation (features are highly discriminative)
- **Region mapping**: Africa (Zenodo), Asia (Kaggle), filterable at inference
- **Soil profiles**: 4 soil types with real NPK/pH distributions from Zenodo data

The model is now integrated into the GUI which supports:
- ✅ Auto-detect GGUF models in LLM/ folder
- ✅ Soil type dropdown → auto-fills NPK/pH
- ✅ Region dropdown → filters predictions
- ✅ Confidence-aware LLM explanations
- ✅ Clean prompt (no "Did you know" forced prefix)